# 05 - Options Pricing

This notebook takes the Ridge model's latest top stock picks and prices one-month call options on them using Black-Scholes and a Cox-Ross-Rubinstein binomial tree.

## Workflow

1. Load daily prices and engineered monthly features.
2. Rebuild the long-format model dataset used by the Ridge classifier.
3. Train on all months before the latest feature month and score the latest month.
4. Pull the one-month Treasury yield from FRED as the risk-free rate.
5. Price 30-day call options with Black-Scholes and a binomial tree.
6. Compare the two prices and visualize the spread.

In [ ]:
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import RidgeClassifier
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

In [ ]:
DATA_DIR = "../data"

features_df = pd.read_csv(f"{DATA_DIR}/features.csv", index_col=0, parse_dates=True)
prices_df = pd.read_csv(f"{DATA_DIR}/prices.csv", index_col=0, parse_dates=True)

print(f"Features shape: {features_df.shape}")
print(f"Prices shape:   {prices_df.shape}")
display(features_df.head(3))

## Recreate the Ridge top picks

The option pricing notebook uses the same six engineered factors as the modeling and backtesting notebooks. To avoid training on the target attached to the latest month, the model trains on months strictly before the latest feature date, then scores the latest month.

In [ ]:
TICKERS = [
    "AAPL", "MSFT", "GOOGL", "AMZN", "META",
    "JPM", "GS", "BAC", "WFC", "C",
    "XOM", "CVX", "COP", "SLB", "EOG",
    "JNJ", "PFE", "UNH", "MRK", "ABT"
]

FEATURE_COLS = ["ret_1m", "ret_3m", "ret_6m", "ma_ratio_20", "ma_ratio_50", "volatility"]

rows = []
for ticker in TICKERS:
    feature_names = [f"{ticker}_{feature}" for feature in FEATURE_COLS]
    target_name = f"{ticker}_target"
    required_cols = feature_names + [target_name]
    if not all(col in features_df.columns for col in required_cols):
        continue

    temp = features_df[required_cols].copy()
    temp.columns = FEATURE_COLS + ["target"]
    temp["ticker"] = ticker
    rows.append(temp)

model_df = pd.concat(rows).sort_index()
print(f"Model dataframe shape: {model_df.shape}")
display(model_df.head())

In [ ]:
latest_signal_date = model_df.index.max()
train = model_df[model_df.index < latest_signal_date].copy()
current = model_df[model_df.index == latest_signal_date].copy()

X_train = train[FEATURE_COLS]
y_train = train["target"]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

ridge = RidgeClassifier(alpha=1.0)
ridge.fit(X_train_scaled, y_train)

current["score"] = ridge.decision_function(scaler.transform(current[FEATURE_COLS]))
top_picks = current.nlargest(5, "score").copy()

print(f"Latest signal date: {latest_signal_date.date()}")
display(top_picks[["ticker", "score", "volatility"]])

## Pull the risk-free rate from FRED

FRED publishes Treasury yields as percentages. This notebook uses `DGS1MO`, the one-month Treasury constant maturity rate, because the option maturity below is set to 30 days. If the FRED request fails, the notebook falls back to a clearly marked manual rate so the rest of the analysis can still run offline.

In [ ]:
def fetch_fred_rate(series_id="DGS1MO", fallback_rate=0.045):
    """Return the latest available FRED annualized rate as a decimal."""
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}"
    try:
        fred = pd.read_csv(url, parse_dates=["observation_date"])
        fred = fred.rename(columns={series_id: "rate"})
        fred["rate"] = pd.to_numeric(fred["rate"], errors="coerce")
        latest = fred.dropna().iloc[-1]
        rate = float(latest["rate"]) / 100
        print(f"Using FRED {series_id} from {latest['observation_date'].date()}: {rate:.3%}")
        return rate, latest["observation_date"], "FRED"
    except Exception as exc:
        print(f"FRED fetch failed: {exc}")
        print(f"Using fallback annualized risk-free rate: {fallback_rate:.3%}")
        return fallback_rate, pd.NaT, "fallback"

risk_free_rate, risk_free_date, risk_free_source = fetch_fred_rate()

## Pricing functions

The Black-Scholes implementation prices European calls. The binomial tree below is also configured for European calls so the comparison is apples-to-apples. For non-dividend-paying European calls, the binomial price should converge toward Black-Scholes as the number of steps rises.

In [ ]:
def normal_cdf(x):
    return 0.5 * (1 + math.erf(x / math.sqrt(2)))


def black_scholes_call(spot, strike, time_to_expiry, risk_free_rate, volatility):
    if time_to_expiry <= 0:
        return max(spot - strike, 0)
    if volatility <= 0:
        forward_intrinsic = spot - strike * math.exp(-risk_free_rate * time_to_expiry)
        return max(forward_intrinsic, 0)

    sqrt_t = math.sqrt(time_to_expiry)
    d1 = (math.log(spot / strike) + (risk_free_rate + 0.5 * volatility**2) * time_to_expiry) / (volatility * sqrt_t)
    d2 = d1 - volatility * sqrt_t
    return spot * normal_cdf(d1) - strike * math.exp(-risk_free_rate * time_to_expiry) * normal_cdf(d2)


def binomial_call(spot, strike, time_to_expiry, risk_free_rate, volatility, steps=100):
    if time_to_expiry <= 0:
        return max(spot - strike, 0)
    if volatility <= 0:
        forward_intrinsic = spot - strike * math.exp(-risk_free_rate * time_to_expiry)
        return max(forward_intrinsic, 0)

    dt = time_to_expiry / steps
    up = math.exp(volatility * math.sqrt(dt))
    down = 1 / up
    discount = math.exp(-risk_free_rate * dt)
    probability = (math.exp(risk_free_rate * dt) - down) / (up - down)

    if not 0 <= probability <= 1:
        raise ValueError("Risk-neutral probability is outside [0, 1]. Increase steps or check inputs.")

    terminal_values = np.array([
        max(spot * (up ** j) * (down ** (steps - j)) - strike, 0)
        for j in range(steps + 1)
    ])

    for _ in range(steps):
        terminal_values = discount * (probability * terminal_values[1:] + (1 - probability) * terminal_values[:-1])

    return float(terminal_values[0])

## Build option inputs

This baseline prices at-the-money calls using each selected stock's latest adjusted close as both spot and strike. Volatility comes from the latest engineered feature row, where it was already annualized in Notebook 2.

In [ ]:
OPTION_DAYS = 30
TIME_TO_EXPIRY = OPTION_DAYS / 365
BINOMIAL_STEPS = 100

latest_price_date = prices_df.index.max()
latest_prices = prices_df.loc[latest_price_date]

option_inputs = []
for _, row in top_picks.iterrows():
    ticker = row["ticker"]
    spot = float(latest_prices[ticker])
    strike = round(spot, 2)
    volatility = float(row["volatility"])

    option_inputs.append({
        "ticker": ticker,
        "model_score": row["score"],
        "spot": spot,
        "strike": strike,
        "volatility": volatility,
        "risk_free_rate": risk_free_rate,
        "time_to_expiry_years": TIME_TO_EXPIRY,
    })

option_inputs_df = pd.DataFrame(option_inputs)
print(f"Latest price date: {latest_price_date.date()}")
display(option_inputs_df)

In [ ]:
pricing_rows = []

for _, option in option_inputs_df.iterrows():
    bs_price = black_scholes_call(
        spot=option["spot"],
        strike=option["strike"],
        time_to_expiry=option["time_to_expiry_years"],
        risk_free_rate=option["risk_free_rate"],
        volatility=option["volatility"],
    )
    tree_price = binomial_call(
        spot=option["spot"],
        strike=option["strike"],
        time_to_expiry=option["time_to_expiry_years"],
        risk_free_rate=option["risk_free_rate"],
        volatility=option["volatility"],
        steps=BINOMIAL_STEPS,
    )

    pricing_rows.append({
        "ticker": option["ticker"],
        "model_score": option["model_score"],
        "spot": option["spot"],
        "strike": option["strike"],
        "volatility": option["volatility"],
        "black_scholes_call": bs_price,
        "binomial_call": tree_price,
        "difference": tree_price - bs_price,
        "abs_difference": abs(tree_price - bs_price),
    })

pricing_df = pd.DataFrame(pricing_rows).sort_values("model_score", ascending=False)
display(pricing_df.style.format({
    "model_score": "{:.3f}",
    "spot": "${:.2f}",
    "strike": "${:.2f}",
    "volatility": "{:.2%}",
    "black_scholes_call": "${:.2f}",
    "binomial_call": "${:.2f}",
    "difference": "${:.4f}",
    "abs_difference": "${:.4f}",
}))

In [ ]:
plot_df = pricing_df.set_index("ticker")[["black_scholes_call", "binomial_call"]]

ax = plot_df.plot(kind="bar", figsize=(10, 5), color=["steelblue", "darkorange"])
ax.set_title("30-Day ATM Call Price: Black-Scholes vs Binomial Tree")
ax.set_ylabel("Option premium")
ax.set_xlabel("Ticker")
ax.grid(axis="y", alpha=0.3)
ax.legend(["Black-Scholes", "Binomial Tree"])
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Interpretation

- The two prices should usually be close because both methods price the same European call under the same assumptions.
- The binomial tree is discrete, so the gap versus Black-Scholes depends on the number of steps. Increasing `BINOMIAL_STEPS` should generally shrink the difference.
- This is a theoretical premium estimate, not a tradable options quote. Real option chains include bid-ask spreads, discrete strikes, dividends, early exercise considerations, transaction costs, and implied volatility rather than historical realized volatility.
- The model picks inherit the same limitations as the earlier notebooks: small sample size, survivorship bias, and no transaction costs.